# Knowledge Catalog entry links — fixture, capability tests, REST and Python SDK

This notebook is the entry-link workstream in one place: it generates the demo fixture,
creates entry links of all four system types, and verifies the capability findings.
Sections 1–3 use the REST API directly; section 6 repeats the same operations with the
`google-cloud-dataplex` Python SDK.

**Findings (verified 2026-08-26).** Entry links are a general-purpose relationship
mechanism with per-type validation; they are not restricted to the business glossary.
Custom entry link types cannot be created: `POST .../entryLinkTypes` returns 404 Method
not found, and no `EntryLinkType` message exists in the public proto or the SDK.

| Type | Direction | Entry-kind rules (observed) |
|---|---|---|
| `related` | symmetric | both ends arbitrary — generic↔generic works |
| `schema-join` | symmetric | both ends arbitrary; requires the `schema-join` aspect on the **link** |
| `definition` | directed | SOURCE arbitrary, TARGET **must** be a glossary term |
| `synonym` | symmetric | both ends **must** be terms; link lives in the `@dataplex` entry group |

Every link has exactly two entry references, whatever its type. The API also enforces
one link per *(type, unordered entry pair)*: creating a duplicate returns 409
AlreadyExists regardless of link ID or reference order.

**Fixture** (created by section 1 in your project): entry group `entry-link-demo` with
generic entries `demo-entry-a`/`demo-entry-b`, glossary `entry-link-demo-glossary` with
terms `demo-term-one`/`demo-term-two`, and one link of each type.

In [ ]:
%pip install -q google-auth requests pandas google-cloud-dataplex

## 0. Config, auth, helpers

Authentication uses Application Default Credentials, with an automatic fallback to the
gcloud CLI token when ADC belongs to a different principal without access to the fixture
project. Term *entry* names are built from the project **number**, with the term's full
resource name as the entry ID — the naming rule that otherwise requires a `searchEntries`
round-trip to discover.

In [ ]:
import json
import subprocess
import time

import requests
import google.auth
from google.auth.transport.requests import Request as AuthRequest

PROJECT_ID = "your-project-id"     # <-- edit
LOCATION = "us-central1"

BASE = "https://dataplex.googleapis.com/v1"
LOC = f"projects/{PROJECT_ID}/locations/{LOCATION}"
EG = f"{LOC}/entryGroups/entry-link-demo"
GLOSSARY = f"{LOC}/glossaries/entry-link-demo-glossary"
ST = "projects/dataplex-types/locations/global/entryLinkTypes"
GENERIC_TYPE = "projects/dataplex-types/locations/global/entryTypes/generic"

PROJECT_NUMBER = subprocess.check_output(
    ["gcloud", "projects", "describe", PROJECT_ID, "--format=value(projectNumber)"],
    text=True).strip()
LOC_PN = f"projects/{PROJECT_NUMBER}/locations/{LOCATION}"
TERM1_ENTRY = f"{LOC_PN}/entryGroups/@dataplex/entries/{LOC_PN}/glossaries/entry-link-demo-glossary/terms/demo-term-one"
TERM2_ENTRY = f"{LOC_PN}/entryGroups/@dataplex/entries/{LOC_PN}/glossaries/entry-link-demo-glossary/terms/demo-term-two"

credentials, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
credentials.refresh(AuthRequest())
_cli_token = None  # set if ADC lacks access (ADC and gcloud CLI can be different principals)


def _auth_header():
    if _cli_token:
        return f"Bearer {_cli_token}"
    if credentials.expired:
        credentials.refresh(AuthRequest())
    return f"Bearer {credentials.token}"


def dataplex_try(method, path, body=None):
    """Never raises: returns (status_code, parsed_body_dict)."""
    resp = requests.request(method, f"{BASE}/{path}",
                            headers={"Authorization": _auth_header(),
                                     "Content-Type": "application/json"},
                            json=body, timeout=60)
    try:
        return resp.status_code, resp.json()
    except ValueError:
        return resp.status_code, {"raw": resp.text[:300]}


def lookup_entry_links(entry_name, link_types=None):
    """All EntryLinks referencing an entry (handles the 10-per-page cap)."""
    links, page_token = [], None
    while True:
        params = [f"entry={entry_name}", "pageSize=10"]
        if link_types:
            params += [f"entryLinkTypes={t}" for t in link_types]
        if page_token:
            params.append(f"pageToken={page_token}")
        code, data = dataplex_try("GET", f"{LOC}:lookupEntryLinks?" + "&".join(params))
        if code != 200:
            raise RuntimeError(data)
        links += data.get("entryLinks", [])
        page_token = data.get("nextPageToken")
        if not page_token:
            return links


def show_links(entry_name):
    for l in lookup_entry_links(entry_name):
        refs = [(r.get("type", "-"), r["name"].rsplit("/", 1)[-1]) for r in l["entryReferences"]]
        print("  ", l["entryLinkType"].rsplit("/", 1)[-1], refs)


code, _ = dataplex_try("GET", f"{LOC}/entryGroups")
if code == 403:
    _cli_token = subprocess.check_output(["gcloud", "auth", "print-access-token"], text=True).strip()
    code, _ = dataplex_try("GET", f"{LOC}/entryGroups")
    print(f"note: ADC principal lacks access to {PROJECT_ID}; using gcloud CLI credentials")
print(f"Access HTTP {code} | {PROJECT_ID} ({PROJECT_NUMBER}) / {LOCATION}")

## 1. Data generation (REST)

Creates the fixture: the entry group, two generic entries, the glossary, and two terms.
Every call is idempotent — an existing resource is reported and skipped. Three API
behaviors are handled here: the system `generic` entry type requires its companion
`dataplex-types.global.generic` aspect; glossary terms require an explicit `parent`
field in the request body (the parent in the URL is not sufficient); and duplicate
terms return 400 INVALID_ARGUMENT rather than 409, so the helper matches on the error
message as well as the status code. Entry group and glossary creation are long-running
operations, hence the short waits after a fresh create.

In [ ]:
def ensure(label, method, path, body=None, settle=0):
    code, data = dataplex_try(method, path, body)
    msg = data.get("error", {}).get("message", "")
    # Quirk: duplicate glossary TERMS return 400 INVALID_ARGUMENT ("already
    # exists") rather than 409, so match on the message as well as the code.
    exists = code == 409 or (code == 400 and "already exists" in msg.lower())
    print(f"{label}: " + ("created" if code == 200 else "already exists" if exists else f"HTTP {code}"))
    if code != 200 and not exists:
        print("   ", msg[:140])
    if code == 200 and settle:
        time.sleep(settle)   # LRO / propagation settle time on fresh create
    return code


ensure("entry group entry-link-demo", "POST",
       f"{LOC}/entryGroups?entryGroupId=entry-link-demo",
       {"description": "Demo group for entry-link scenario testing"}, settle=8)

for e in ["demo-entry-a", "demo-entry-b"]:
    ensure(f"entry {e}", "POST", f"{EG}/entries?entryId={e}", {
        "entryType": GENERIC_TYPE,
        "entrySource": {"description": f"Demo entry {e} for entry-link testing"},
        "aspects": {"dataplex-types.global.generic": {
            "data": {"type": "demo", "system": "entry-link-demo"}}},
    })

ensure("glossary entry-link-demo-glossary", "POST",
       f"{LOC}/glossaries?glossaryId=entry-link-demo-glossary",
       {"displayName": "Entry Link Demo Glossary",
        "description": "Terms for entry-link scenario testing"}, settle=12)

ensure("term demo-term-one", "POST", f"{GLOSSARY}/terms?termId=demo-term-one",
       {"parent": GLOSSARY, "displayName": "Demo Term One",
        "description": "Business definition used by the definition-link scenario."})
ensure("term demo-term-two", "POST", f"{GLOSSARY}/terms?termId=demo-term-two",
       {"parent": GLOSSARY, "displayName": "Demo Term Two",
        "description": "Synonym of Demo Term One."})

## 2. Create the entry links — explicit REST calls

Four `POST {entryGroup}/entryLinks?entryLinkId=...` requests with full payloads. The call
is identical for every type; what varies is the type name, whether the references are
symmetric (`UNSPECIFIED`) or directed (`SOURCE`/`TARGET`), and the aspect payload that
`schema-join` requires. Re-running reports `already exists`: uniqueness is enforced on
the *(type, unordered entry pair)*, not the link ID.

In [ ]:
def create_entry_link(entry_group, link_id, body):
    code, data = dataplex_try("POST", f"{entry_group}/entryLinks?entryLinkId={link_id}", body)
    print(f"{link_id}: " + {200: "created", 409: "already exists"}.get(code, f"HTTP {code}"))
    if code not in (200, 409):
        print("   ", data.get("error", {}).get("message", "")[:140])


ENTRY_A = f"{EG}/entries/demo-entry-a"
ENTRY_B = f"{EG}/entries/demo-entry-b"

# 1. related — the minimal entry<->entry link: a type and two symmetric references.
create_entry_link(EG, "demo-related", {
    "entryLinkType": f"{ST}/related",
    "entryReferences": [
        {"name": ENTRY_A, "type": "UNSPECIFIED"},
        {"name": ENTRY_B, "type": "UNSPECIFIED"},
    ],
})

# 2. schema-join — same shape PLUS the required Dataplex-owned aspect on the link;
#    the joins payload is where the a.id = b.a_id semantics live.
create_entry_link(EG, "demo-schema-join", {
    "entryLinkType": f"{ST}/schema-join",
    "aspects": {"dataplex-types.global.schema-join": {"data": {
        "userManaged": True,
        "joins": [{
            "source": {"name": "demo_entry_a", "fields": ["id"]},
            "target": {"name": "demo_entry_b", "fields": ["a_id"]},
            "type": "JOIN",              # or FOREIGN_KEY
            "inferenceSource": "USER",   # USER | TABLE_CONSTRAINTS | QUERY_HISTORY | AGENT
        }],
    }}},
    "entryReferences": [
        {"name": ENTRY_A, "type": "UNSPECIFIED"},
        {"name": ENTRY_B, "type": "UNSPECIFIED"},
    ],
})

# 3. definition — directed: SOURCE can be any entry, TARGET must be a glossary term.
create_entry_link(EG, "demo-definition", {
    "entryLinkType": f"{ST}/definition",
    "entryReferences": [
        {"name": ENTRY_A, "type": "SOURCE"},
        {"name": TERM1_ENTRY, "type": "TARGET"},
    ],
})

# 4. synonym — term<->term only; created in the @dataplex entry group (project-NUMBER form).
create_entry_link(f"{LOC_PN}/entryGroups/@dataplex", "demo-synonym", {
    "entryLinkType": f"{ST}/synonym",
    "entryReferences": [
        {"name": TERM1_ENTRY, "type": "UNSPECIFIED"},
        {"name": TERM2_ENTRY, "type": "UNSPECIFIED"},
    ],
})

## 3. Inspect the link graph

`lookupEntryLinks` returns all three links on the asset entry. The other read surfaces
show less: the console renders only the glossary attachment, and `lookupContext` folds
only the attached term into its output (the `terms` field). An agent that needs the
related entries or the join metadata must call `lookupEntryLinks` itself — this is why
the agent workstream chains lookups rather than relying on one `lookupContext` call.

In [ ]:
print("Links on demo-entry-a (asset side):")
show_links(ENTRY_A)

print("\nLinks on demo-term-one (term side):")
show_links(TERM1_ENTRY)

code, data = dataplex_try("POST", f"{LOC}:lookupContext", {
    "resources": [ENTRY_A], "options": {"format": "json"}})
print(f"\nlookupContext on demo-entry-a (HTTP {code}):")
print(data.get("context", data)[:600] if code == 200 else data)

## 4. Capability probes — re-runnable PASS/FAIL matrix

Each probe attempts a link create with a random ID and asserts the expected outcome:
success for supported shapes, rejection for unsupported ones. A 409 AlreadyExists counts
as supported, since a duplicate of a standing fixture link still proves the shape is
allowed. Probes delete anything they create, so the cell repeats cleanly and leaves the
fixture unchanged.

In [ ]:
import uuid

def probe(label, expect_ok, entry_group, body):
    link_id = f"probe-{uuid.uuid4().hex[:6]}"
    code, data = dataplex_try("POST", f"{entry_group}/entryLinks?entryLinkId={link_id}", body)
    ok = code == 200 or (code == 409 and "already exists" in
                         data.get("error", {}).get("message", "").lower())
    print(f"{'PASS' if ok == expect_ok else 'FAIL'}  [{code}] {label}")
    if code not in (200, 409):
        print("        " + data.get("error", {}).get("message", "")[:120])
    if code == 200:
        dataplex_try("DELETE", f"{entry_group}/entryLinks/{link_id}")
    return ok == expect_ok


results = [
    probe("related: generic <-> generic (arbitrary entries)", True, EG, {
        "entryLinkType": f"{ST}/related",
        "entryReferences": [{"name": ENTRY_A, "type": "UNSPECIFIED"}, {"name": ENTRY_B, "type": "UNSPECIFIED"}]}),
    probe("synonym: generic <-> generic (must be terms)", False, EG, {
        "entryLinkType": f"{ST}/synonym",
        "entryReferences": [{"name": ENTRY_A, "type": "UNSPECIFIED"}, {"name": ENTRY_B, "type": "UNSPECIFIED"}]}),
    probe("definition: generic -> generic (TARGET must be a term)", False, EG, {
        "entryLinkType": f"{ST}/definition",
        "entryReferences": [{"name": ENTRY_A, "type": "SOURCE"}, {"name": ENTRY_B, "type": "TARGET"}]}),
    probe("definition: asset -> real term", True, EG, {
        "entryLinkType": f"{ST}/definition",
        "entryReferences": [{"name": ENTRY_A, "type": "SOURCE"}, {"name": TERM1_ENTRY, "type": "TARGET"}]}),
    probe("related with SOURCE/TARGET refs (directionality enforced)", False, EG, {
        "entryLinkType": f"{ST}/related",
        "entryReferences": [{"name": ENTRY_A, "type": "SOURCE"}, {"name": ENTRY_B, "type": "TARGET"}]}),
    probe("custom link type (not supported by the API)", False, EG, {
        "entryLinkType": f"projects/{PROJECT_ID}/locations/global/entryLinkTypes/depends-on",
        "entryReferences": [{"name": ENTRY_A, "type": "SOURCE"}, {"name": ENTRY_B, "type": "TARGET"}]}),
]

print(f"\n{sum(results)}/{len(results)} scenarios behaved as expected")

## 5. Recorded test matrix

This table is the static record of the verified runs (August 2026, verified against two
Google Cloud test projects). It makes no API calls,
so the results display without credentials. The last row is informational rather than
pass/fail: system link types are readable resources gated by `dataplex.entryLinkTypes.get`,
a permission no predefined Dataplex role grants.

In [ ]:
import pandas as pd

test_matrix = pd.DataFrame([
    ("related: generic <-> generic",               "related",     "allowed",  "200 (409 AlreadyExists on re-run)",         "PASS"),
    ("related: custom-entryType entry <-> generic", "related",    "allowed",  "200",                                       "PASS"),
    ("schema-join: generic <-> generic + aspect",  "schema-join", "allowed",  "200",                                       "PASS"),
    ("schema-join: without joins aspect",          "schema-join", "rejected", "400 missing required schema-join aspect",   "PASS"),
    ("definition: asset SOURCE -> term TARGET",    "definition",  "allowed",  "200",                                       "PASS"),
    ("definition: generic -> generic (no term)",   "definition",  "rejected", "400 TARGET invalid for link type",          "PASS"),
    ("definition: term SOURCE -> asset TARGET",    "definition",  "rejected", "400 reversed direction invalid",            "PASS"),
    ("synonym: term <-> term (@dataplex group)",   "synonym",     "allowed",  "200",                                       "PASS"),
    ("synonym: generic <-> generic",               "synonym",     "rejected", "400 entries invalid for link type",         "PASS"),
    ("synonym: term <-> generic (mixed)",          "synonym",     "rejected", "400 generic end invalid",                   "PASS"),
    ("related with SOURCE/TARGET refs",            "related",     "rejected", "400 directionality enforced per type",      "PASS"),
    ("reference custom link type (own project)",   "custom",      "rejected", "403 type does not exist",                   "PASS"),
    ("POST .../entryLinkTypes (create type)",      "custom",      "rejected", "404 Method not found (no CRUD surface)",    "PASS"),
    ("GET system entryLinkType definition",        "system",      "n/a",      "403 needs dataplex.entryLinkTypes.get",     "INFO"),
], columns=["scenario", "link_type", "expected", "observed", "result"])

print(f"{(test_matrix.result == 'PASS').sum()}/{(test_matrix.result != 'INFO').sum()} scenarios behaved as expected")
test_matrix

## 6. Same functionality — `google-cloud-dataplex` Python SDK

Sections 6a–6c repeat sections 1–3 with the Python SDK. `CatalogServiceClient` covers
entry groups, entries, entry links, `lookup_entry_links`, and `lookup_context`;
`BusinessGlossaryServiceClient` covers glossaries and terms. Differences from the REST
surface:

- The SDK defines no `EntryLinkType` message or CRUD methods — the same absence as the
  proto and REST surfaces.
- `EntryReference` exposes the reference type as `type_` (trailing underscore; `type`
  is reserved).
- Aspects use the same map keys (`dataplex-types.global.generic`), and `Aspect.data`
  accepts a plain dict.
- Entry group and glossary creates return long-running operations; `.result()` waits
  for completion.
- Every cell is idempotent: `AlreadyExists`, and the 400 "already exists" variant that
  duplicate terms raise, are caught and reported.

In [ ]:
from google.api_core import exceptions as gexc
from google.cloud import dataplex_v1
from google.oauth2.credentials import Credentials

# Same credential fallback as the REST helpers (gcloud CLI token if ADC lacks access).
# CLI tokens expire in ~1h; re-run this cell if calls start failing with 401.
sdk_creds = Credentials(token=_cli_token) if _cli_token else credentials
catalog = dataplex_v1.CatalogServiceClient(credentials=sdk_creds)
glossary_client = dataplex_v1.BusinessGlossaryServiceClient(credentials=sdk_creds)

def ensure_sdk(label, fn):
    try:
        result = fn()
        print(f"{label}: created")
        return result
    except gexc.AlreadyExists:
        print(f"{label}: already exists")
    except gexc.InvalidArgument as e:
        # Duplicate glossary terms surface as 400 INVALID_ARGUMENT, not 409.
        if "already exists" in str(e).lower():
            print(f"{label}: already exists")
        else:
            raise

print("clients ready")

In [ ]:
# --- 6a. Data generation via SDK (parity with section 1) ---

ensure_sdk("entry group entry-link-demo", lambda: catalog.create_entry_group(
    parent=LOC, entry_group_id="entry-link-demo",
    entry_group=dataplex_v1.EntryGroup(description="Demo group for entry-link scenario testing"),
).result())

for e in ["demo-entry-a", "demo-entry-b"]:
    ensure_sdk(f"entry {e}", lambda e=e: catalog.create_entry(
        parent=EG, entry_id=e,
        entry=dataplex_v1.Entry(
            entry_type=GENERIC_TYPE,
            entry_source=dataplex_v1.EntrySource(description=f"Demo entry {e} for entry-link testing"),
            aspects={"dataplex-types.global.generic": dataplex_v1.Aspect(
                data={"type": "demo", "system": "entry-link-demo"})},
        )))

ensure_sdk("glossary entry-link-demo-glossary", lambda: glossary_client.create_glossary(
    parent=LOC, glossary_id="entry-link-demo-glossary",
    glossary=dataplex_v1.Glossary(display_name="Entry Link Demo Glossary",
                                  description="Terms for entry-link scenario testing"),
).result())

for term_id, name, desc in [
    ("demo-term-one", "Demo Term One", "Business definition used by the definition-link scenario."),
    ("demo-term-two", "Demo Term Two", "Synonym of Demo Term One."),
]:
    ensure_sdk(f"term {term_id}", lambda t=term_id, n=name, d=desc: glossary_client.create_glossary_term(
        parent=GLOSSARY, term_id=t,
        term=dataplex_v1.GlossaryTerm(parent=GLOSSARY, display_name=n, description=d)))

In [ ]:
# --- 6b. Entry link creation via SDK (parity with section 2) ---

Ref = dataplex_v1.EntryLink.EntryReference

def sdk_link(entry_group, link_id, link_type, references, aspects=None):
    ensure_sdk(f"link {link_id}", lambda: catalog.create_entry_link(
        parent=entry_group, entry_link_id=link_id,
        entry_link=dataplex_v1.EntryLink(
            entry_link_type=link_type,
            entry_references=references,
            aspects=aspects or {},
        )))

sdk_link(EG, "demo-related", f"{ST}/related", [
    Ref(name=ENTRY_A, type_=Ref.Type.UNSPECIFIED),
    Ref(name=ENTRY_B, type_=Ref.Type.UNSPECIFIED),
])

sdk_link(EG, "demo-schema-join", f"{ST}/schema-join", [
    Ref(name=ENTRY_A, type_=Ref.Type.UNSPECIFIED),
    Ref(name=ENTRY_B, type_=Ref.Type.UNSPECIFIED),
], aspects={"dataplex-types.global.schema-join": dataplex_v1.Aspect(data={
    "userManaged": True,
    "joins": [{
        "source": {"name": "demo_entry_a", "fields": ["id"]},
        "target": {"name": "demo_entry_b", "fields": ["a_id"]},
        "type": "JOIN",
        "inferenceSource": "USER",
    }],
})})

sdk_link(EG, "demo-definition", f"{ST}/definition", [
    Ref(name=ENTRY_A, type_=Ref.Type.SOURCE),
    Ref(name=TERM1_ENTRY, type_=Ref.Type.TARGET),
])

sdk_link(f"{LOC_PN}/entryGroups/@dataplex", "demo-synonym", f"{ST}/synonym", [
    Ref(name=TERM1_ENTRY, type_=Ref.Type.UNSPECIFIED),
    Ref(name=TERM2_ENTRY, type_=Ref.Type.UNSPECIFIED),
])

In [ ]:
# --- 6c. Lookups via SDK (parity with section 3) ---

print("lookup_entry_links on demo-entry-a:")
resp = catalog.lookup_entry_links(request=dataplex_v1.LookupEntryLinksRequest(
    name=LOC, entry=ENTRY_A))
for l in resp.entry_links:
    print("  ", l.entry_link_type.rsplit("/", 1)[-1],
          [(r.type_.name, r.name.rsplit("/", 1)[-1]) for r in l.entry_references])

print("\nlookup_context on demo-entry-a:")
ctx = catalog.lookup_context(request=dataplex_v1.LookupContextRequest(
    name=LOC, resources=[ENTRY_A], options={"format": "json"}))
print(ctx.context[:600])

## 7. Teardown (guarded)

Set `TEARDOWN = True` and run to remove the fixture in dependency order: links, then
entries, then the entry group, then terms, then the glossary.

In [ ]:
TEARDOWN = False

if TEARDOWN:
    for path in [
        f"{EG}/entryLinks/demo-related",
        f"{EG}/entryLinks/demo-schema-join",
        f"{EG}/entryLinks/demo-definition",
        f"{LOC_PN}/entryGroups/@dataplex/entryLinks/demo-synonym",
        f"{EG}/entries/demo-entry-a",
        f"{EG}/entries/demo-entry-b",
        EG,
        f"{GLOSSARY}/terms/demo-term-one",
        f"{GLOSSARY}/terms/demo-term-two",
        GLOSSARY,
    ]:
        code, _ = dataplex_try("DELETE", path)
        print(code, "DELETE", path.rsplit("/", 2)[-1] if "/" in path else path)
else:
    print("TEARDOWN is False — fixture left standing")